# ML #2 — Winter Gas Storage Stress Classifier
**Zeus — Energy Security Index | Phase II POC**

**ML type:** supervised binary classification 

**Label:** `storage_stress = 1` if a country's gas storage drops below 30% full during winter (Nov 1–Mar 31), else `0`.

**Data:** GIE AGSI API - daily country-level gas storage. One row = one (country, winter) pair. 17 countries, winters 2015–2024.

**Pipeline:** API → clean → CSV → EDA → viz → model → feature importance.

**Tasks remaining:** "stress" is a proxy for supply-shock vulnerability; features are storage-only (Eurostat import dependence planned); small sample (~200 rows) → simple models only.

In [4]:
import requests
import pandas as pd
import numpy as np
import time
from datetime import date

In [5]:
import os
from dotenv import load_dotenv

load_dotenv("apsi.env")   # .env must be in the same folder as this notebook
AGSI_API_KEY = os.getenv("AGSI_API_KEY")
assert AGSI_API_KEY, "AGSI_API_KEY not found — check your .env file"

STRESS_THRESHOLD = 30.0
WINTER_START_MONTH = 11
WINTER_END_MONTH = 3
COUNTRIES = ["DE", "FR", "IT", "NL", "BE", "AT", "ES", "PL", "CZ", "SK",
             "HU", "RO", "BG", "DK", "PT", "HR", "LV"]
START_YEAR = 2015
END_YEAR = 2024

In [7]:
# Pull all the storage data for one country, page by page
def fetch_country(country, year_from, year_to, api_key):
    rows, page = [], 1
    while True:
        r = requests.get(
            "https://agsi.gie.eu/api",
            params={"country": country, "from": f"{year_from}-01-01",
                    "to": f"{year_to}-12-31", "page": page, "size": 300},
            headers={"x-key": api_key},
        )
        r.raise_for_status()
        payload = r.json()
        data = payload.get("data", [])
        if not data:
            break
        rows.extend(data)
        if page >= int(payload.get("last_page", page)):
            break
        page += 1
        time.sleep(0.3)
    df = pd.DataFrame(rows)
    df["country"] = country
    return df

# The AGSI server sometimes hiccups, so retry a few times before giving up
def fetch_with_retry(country, year_from, year_to, api_key, attempts=4):
    for attempt in range(1, attempts + 1):
        try:
            return fetch_country(country, year_from, year_to, api_key)
        except Exception as e:
            wait = attempt * 5   # wait longer each time
            print(f"  {country} attempt {attempt} failed ({e}); retrying in {wait}s")
            time.sleep(wait)
    print(f"  {country} FAILED after {attempts} attempts")
    return None

# Loop through every country, being polite to the server between each
frames = []
for c in COUNTRIES:
    print(f"Fetching {c}...")
    df = fetch_with_retry(c, START_YEAR - 1, END_YEAR + 1, AGSI_API_KEY)
    if df is not None:
        frames.append(df)
    time.sleep(2)

raw = pd.concat(frames, ignore_index=True)
print("\nTotal rows:", raw.shape)
print(raw["country"].value_counts())

Fetching DE...
Fetching FR...
Fetching IT...
Fetching NL...
Fetching BE...
Fetching AT...
Fetching ES...
Fetching PL...
Fetching CZ...
Fetching SK...
Fetching HU...
Fetching RO...
Fetching BG...
Fetching DK...
Fetching PT...
Fetching HR...
Fetching LV...

Total rows: (71628, 23)
country
DE    4383
SK    4383
HR    4383
PT    4383
DK    4383
BG    4383
RO    4383
HU    4383
CZ    4383
FR    4383
PL    4383
ES    4383
AT    4383
BE    4383
NL    4383
IT    4383
LV    1500
Name: count, dtype: int64


In [8]:
# Quick sanity check — what columns, how much per country, anything missing?
print("Columns:", list(raw.columns))
print("\nRows per country:")
print(raw["country"].value_counts())
print("\nMissing values:")
print(raw.isna().sum())
raw.head(10)

Columns: ['name', 'code', 'url', 'updatedAt', 'gasDayStart', 'gasDayEnd', 'gasInStorage', 'consumption', 'consumptionFull', 'injection', 'withdrawal', 'netWithdrawal', 'workingGasVolume', 'injectionCapacity', 'withdrawalCapacity', 'contractedCapacity', 'availableCapacity', 'coveredCapacity', 'status', 'trend', 'full', 'info', 'country']

Rows per country:
country
DE    4383
SK    4383
HR    4383
PT    4383
DK    4383
BG    4383
RO    4383
HU    4383
CZ    4383
FR    4383
PL    4383
ES    4383
AT    4383
BE    4383
NL    4383
IT    4383
LV    1500
Name: count, dtype: int64

Missing values:
name                  0
code                  0
url                   0
updatedAt             0
gasDayStart           0
gasDayEnd             0
gasInStorage          0
consumption           0
consumptionFull       0
injection             0
withdrawal            0
netWithdrawal         0
workingGasVolume      0
injectionCapacity     0
withdrawalCapacity    0
contractedCapacity    0
availableCapacity   

,name,code,url,updatedAt,gasDayStart,gasDayEnd,gasInStorage,consumption,consumptionFull,injection,...,injectionCapacity,withdrawalCapacity,contractedCapacity,availableCapacity,coveredCapacity,status,trend,full,info,country
0,Germany,DE,DE,2026-02-09 07:59:31,2025-12-31,2026-01-01,142.1116,903.9000,15.72,18,...,4211.2,7086.73,203.1733,48.065,-,C,-0.53,56.59,[],DE
1,Germany,DE,DE,2026-01-21 09:21:32,2025-12-30,2025-12-31,143.446,903.9000,15.87,18.95,...,4211.2,7086.73,203.1733,48.065,-,C,-0.56,57.12,[],DE
2,Germany,DE,DE,2026-01-02 11:25:50,2025-12-29,2025-12-30,144.8559,903.9000,16.03,17.4,...,4211.2,7086.73,203.1734,48.065,-,E,-0.61,57.68,[],DE
3,Germany,DE,DE,2026-01-01 11:25:02,2025-12-28,2025-12-29,146.3856,903.9000,16.19,11.88,...,4211.2,7086.73,203.1734,48.065,-,E,-0.43,58.29,[],DE
4,Germany,DE,DE,2026-01-02 07:31:56,2025-12-27,2025-12-28,147.4637,903.9000,16.31,31.26,...,4211.2,7086.73,203.1734,48.065,-,C,-0.42,58.72,[],DE
5,Germany,DE,DE,2026-01-02 07:24:51,2025-12-26,2025-12-27,148.5261,903.9000,16.43,10.1,...,4211.2,7086.73,203.1734,48.065,-,C,-0.64,59.14,[],DE
6,Germany,DE,DE,2025-12-29 11:25:05,2025-12-25,2025-12-26,150.122,903.9000,16.61,10.4,...,4211.2,7086.73,203.1734,48.065,-,C,-0.52,59.78,[],DE
7,Germany,DE,DE,2025-12-28 11:25:05,2025-12-24,2025-12-25,151.4402,903.9000,16.75,24.48,...,4211.2,7086.73,203.1734,48.065,-,C,-0.32,60.3,[],DE
8,Germany,DE,DE,2025-12-27 11:25:05,2025-12-23,2025-12-24,152.2448,903.9000,16.84,22.09,...,4211.2,7086.73,203.1734,48.065,-,C,-0.25,60.62,[],DE
9,Germany,DE,DE,2025-12-26 11:25:05,2025-12-22,2025-12-23,152.8778,903.9000,16.91,9.31,...,4211.2,7086.73,203.1734,48.065,-,C,-0.23,60.87,[],DE


In [9]:
# Save raw data so you don't have to re-hit the API every time
raw.to_csv("agsi_raw.csv", index=False)
print("Saved agsi_raw.csv —", raw.shape[0], "rows")

Saved agsi_raw.csv — 71628 rows


In [10]:
# Make sure dates are real dates and storage % is a real number
raw["gasDayStart"] = pd.to_datetime(raw["gasDayStart"], errors="coerce")
raw["full"] = pd.to_numeric(raw["full"], errors="coerce")

# Drop rows missing the stuff we actually need, keep just the useful columns
clean = raw.dropna(subset=["gasDayStart", "full", "country"]).copy()
clean = clean[["country", "gasDayStart", "full", "gasInStorage", "trend"]]

print("Rows after cleaning:", clean.shape[0])
print("\nRows per country:")
print(clean["country"].value_counts())

clean.to_csv("agsi_clean.csv", index=False)
print("\nSaved agsi_clean.csv")

Rows after cleaning: 71538

Rows per country:
country
DE    4383
FR    4383
PT    4383
DK    4383
BG    4383
RO    4383
HU    4383
SK    4383
CZ    4383
PL    4383
ES    4383
AT    4383
BE    4383
NL    4383
IT    4383
HR    4293
LV    1500
Name: count, dtype: int64

Saved agsi_clean.csv


In [11]:
# One-variable look: overall spread of storage levels
print("Storage % full — overall summary:")
print(clean["full"].describe().round(2))

# Two-variable look: which countries run fuller / emptier on average
print("\nMean storage % by country:")
print(clean.groupby("country")["full"].mean().round(1).sort_values())

Storage % full — overall summary:
count    71538.00
mean        65.30
std         23.65
min          0.00
25%         47.20
50%         68.04
75%         85.54
max        126.04
Name: full, dtype: float64

Mean storage % by country:
country
LV    53.6
HU    55.9
RO    56.6
BG    60.1
BE    60.5
FR    60.8
HR    62.8
SK    63.1
AT    63.7
NL    64.0
DK    67.5
DE    68.1
CZ    68.3
IT    71.7
PL    73.3
PT    76.2
ES    76.2
Name: full, dtype: float64


In [13]:
import plotly.express as px
import os
os.makedirs("figures", exist_ok=True)

# Plot storage over time for a few countries, with the stress line marked
plot_df = clean[clean["country"].isin(["DE", "FR", "IT", "NL"])]
fig = px.line(plot_df, x="gasDayStart", y="full", color="country",
              title="EU Gas Storage Levels Over Time",
              labels={"full": "% full", "gasDayStart": "Date"})
fig.add_hline(y=30, line_dash="dash", line_color="red",
              annotation_text="30% stress threshold")
fig.write_html("figures/storage_history.html")
fig.show()

In [14]:
# === Build the modeling dataset ===
clean["year"] = clean["gasDayStart"].dt.year
clean["month"] = clean["gasDayStart"].dt.month

def assign_winter(row):
    if row["month"] >= WINTER_START_MONTH:
        return row["year"]
    elif row["month"] <= WINTER_END_MONTH:
        return row["year"] - 1
    return np.nan

clean["winter"] = clean.apply(assign_winter, axis=1)

winter_rows = clean[clean["winter"].notna()]
label = (winter_rows.groupby(["country", "winter"])
         .agg(min_winter_full=("full", "min"),
              days=("full", "count"))
         .reset_index())
label = label[label["days"] >= 90]
label["storage_stress"] = (label["min_winter_full"] < STRESS_THRESHOLD).astype(int)

def pre_winter(country, winter_year):
    cutoff = pd.Timestamp(year=int(winter_year), month=11, day=1)
    hist = clean[(clean["country"] == country) & (clean["gasDayStart"] < cutoff)]
    octw = hist[hist["gasDayStart"] >= cutoff - pd.Timedelta(days=31)]
    if len(octw) < 2:
        return None
    octw = octw.sort_values("gasDayStart")
    return {
        "country": country,
        "winter": winter_year,
        "storage_at_start": octw["full"].mean(),
        "storage_trend_30d": octw["full"].iloc[-1] - octw["full"].iloc[0],
        "storage_volatility": hist["full"].tail(90).std(),
    }

feats = [pre_winter(r.country, r.winter) for r in label.itertuples()]
feats = pd.DataFrame([f for f in feats if f is not None])

dataset = label.merge(feats, on=["country", "winter"]).dropna()
print("Final dataset shape:", dataset.shape)
print("\nLabel balance (0 = no stress, 1 = stress):")
print(dataset["storage_stress"].value_counts())
dataset.head()

Final dataset shape: (179, 8)

Label balance (0 = no stress, 1 = stress):
storage_stress
0    101
1     78
Name: count, dtype: int64


,country,winter,min_winter_full,days,storage_stress,storage_at_start,storage_trend_30d,storage_volatility
0,AT,2014.0,16.49,151,1,95.677419,-1.76,6.013985
1,AT,2015.0,36.17,152,0,74.723226,1.55,6.013985
2,AT,2016.0,14.59,151,1,89.321935,-2.89,6.013985
3,AT,2017.0,14.47,151,1,84.642903,5.02,6.013985
4,AT,2018.0,44.00,151,0,72.657097,6.74,6.013985


In [15]:
# === Plot: Lowest winter storage by country ===
worst = (label.groupby("country")["min_winter_full"].min()
         .sort_values().reset_index())
worst["color"] = worst["min_winter_full"].apply(
    lambda v: "stress" if v < STRESS_THRESHOLD else "ok")

fig = px.bar(worst, x="country", y="min_winter_full", color="color",
             color_discrete_map={"stress": "red", "ok": "steelblue"},
             title="Lowest Winter Storage Ever Recorded, by Country",
             labels={"min_winter_full": "% full"})
fig.add_hline(y=30, line_dash="dash", line_color="red")
fig.write_html("figures/min_storage_by_country.html")
fig.show()

In [16]:
# === Plot: Pre-winter storage vs winter minimum ===
plot_data = dataset.copy()
plot_data["outcome"] = plot_data["storage_stress"].map({0: "No stress", 1: "Stress"})

fig = px.scatter(plot_data, x="storage_at_start", y="min_winter_full",
                 color="outcome",
                 color_discrete_map={"No stress": "steelblue", "Stress": "red"},
                 hover_data=["country", "winter"],
                 title="Does a Full Start Mean a Safe Winter?",
                 labels={"storage_at_start": "Storage % at start of winter",
                         "min_winter_full": "Minimum storage % during winter"})
fig.add_hline(y=30, line_dash="dash", line_color="red")
fig.write_html("figures/start_vs_min.html")
fig.show()

In [17]:
# === Train POC models ===
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix

features = ["storage_at_start", "storage_trend_30d", "storage_volatility"]
X = dataset[features]
y = dataset["storage_stress"]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in [
    ("Logistic Regression", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ("Random Forest", RandomForestClassifier(n_estimators=200, max_depth=4,
                                             class_weight="balanced", random_state=42)),
]:
    preds = cross_val_predict(model, X, y, cv=cv)
    print(f"\n=== {name} ===")
    print(confusion_matrix(y, preds))
    print(classification_report(y, preds))


=== Logistic Regression ===
[[76 25]
 [36 42]]
              precision    recall  f1-score   support

           0       0.68      0.75      0.71       101
           1       0.63      0.54      0.58        78

    accuracy                           0.66       179
   macro avg       0.65      0.65      0.65       179
weighted avg       0.66      0.66      0.66       179


=== Random Forest ===
[[75 26]
 [23 55]]
              precision    recall  f1-score   support

           0       0.77      0.74      0.75       101
           1       0.68      0.71      0.69        78

    accuracy                           0.73       179
   macro avg       0.72      0.72      0.72       179
weighted avg       0.73      0.73      0.73       179



In [19]:
# === Feature importance ===
rf = RandomForestClassifier(n_estimators=200, max_depth=4,
                            class_weight="balanced", random_state=42)
rf.fit(X, y)
imp = (pd.Series(rf.feature_importances_, index=features)
       .sort_values().reset_index())
imp.columns = ["feature", "importance"]

fig = px.bar(imp, x="importance", y="feature", orientation="h",
             title="What drives winter storage stress")
fig.write_html("figures/feature_importance.html")
fig.show()
print(imp)

              feature  importance
0   storage_trend_30d    0.261804
1  storage_volatility    0.350973
2    storage_at_start    0.387223
